# 03 Nonseasonal ARIMA Models in Python

By the end of this notebook, you should be able to:

- write AR, MA, ARMA, and ARIMA model forms;
- understand how differencing order `d` connects the working series to the original series;
- decide whether a constant or drift term is appropriate;
- fit candidate ARIMA models with `statsmodels` and compare them responsibly.

In [1]:
from lite_setup import ensure_packages
await ensure_packages()

Running outside JupyterLite; assuming packages are already installed.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from checks import check, check_between, check_close, check_columns
from boxjenkins_utils import (
    first_difference, second_difference, seasonal_difference, regular_then_seasonal_difference,
    acf_pacf_table, mean_zero_t_test, fit_arima, fit_sarima, parameter_table,
    forecast_table, ljung_box_table, arima_grid_search, plot_series,
    plot_acf_pacf_pair, plot_forecast,
)
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.precision', 4)
DATA_DIR = Path('data')
towel = pd.read_csv(DATA_DIR / 'paper_towel_sales.csv')
y = towel['Towel_Sales']
z = first_difference(y)

## Model Forms

For a stationary working series $z_t$:

MA(q):

$$z_t = \delta + a_t - \theta_1a_{t-1}-\cdots-\theta_q a_{t-q}.$$

AR(p):

$$z_t = \delta + \phi_1z_{t-1}+\cdots+\phi_pz_{t-p}+a_t.$$

ARMA(p, q) combines both parts. If the original series $y_t$ must be differenced $d$ times to obtain $z_t$, the model is ARIMA(p, d, q) for $y_t$.

## Backshift Notation

The backshift operator is a compact way to write ARIMA models:

$$B y_t = y_{t-1}, \quad B^k y_t = y_{t-k}.$$

A first difference is

$$(1-B)y_t = y_t-y_{t-1},$$

and a second difference is

$$(1-B)^2y_t = y_t-2y_{t-1}+y_{t-2}.$$

The nonseasonal ARIMA model can be written as

$$\phi(B)(1-B)^d y_t = \delta + \theta(B)a_t,$$

where $p$ is the order of $\phi(B)$, $d$ is the number of regular differences, and $q$ is the order of $\theta(B)$.

## A Sign Convention Warning

The lecture and textbook write the MA(1) model as

$$z_t = \delta + a_t - \theta_1 a_{t-1}.$$

`statsmodels` reports MA parameters in the form

$$z_t = \delta + a_t + \psi_1 a_{t-1}.$$

Therefore, approximately, $\psi_1=-\theta_1$. A negative `ma.L1` in `statsmodels` corresponds to a positive textbook $\theta_1$.

## Constant Term Or Drift?

For the stationary working series:

- If the working series is the original series and its mean is not close to 0, include a constant/intercept.
- If the working series is a differenced series and its mean is not close to 0, that indicates drift in the original series.
- If the working-series mean is close to 0, do not force a constant.

In practice, combine subject-matter reasoning, a mean-zero t test, AIC/BIC, and diagnostics. The t test is a guide, not an automatic model selector.

## What The Constant Means

For an MA(q) model written on a stationary working series, the constant is the working-series mean: $\delta=\mu$.

For an AR(p) model,

$$z_t = \delta + \phi_1 z_{t-1}+\cdots+\phi_p z_{t-p}+a_t,$$

and the stationary mean satisfies

$$\mu = \frac{\delta}{1-\phi_1-\cdots-\phi_p}.$$

Equivalently, $\delta=\mu(1-\phi_1-\cdots-\phi_p)$. If the working series is a difference, a nonzero mean in the differences is drift in the original series.

In [3]:
mean_zero_t_test(z).round(4)

n          119.0000
mean         0.0054
sd           1.1041
se           0.1012
t            0.0536
p_value      0.9574
dtype: float64

For the towel first differences, the mean is small relative to the variation, so the lecture workflow usually fits ARIMA(0, 1, 1) without drift.

In [4]:
ma1 = fit_arima(y, order=(0, 1, 1), trend='n')
parameter_table(ma1).round(4)

,estimate,std_error,z_or_t,p_value
ma.L1,0.3518,0.0745,4.7220,0.0
sigma2,1.0708,0.1323,8.0928,0.0


In [5]:
print(ma1.summary())

                               SARIMAX Results                                
Dep. Variable:            Towel_Sales   No. Observations:                  120
Model:                 ARIMA(0, 1, 1)   Log Likelihood                -172.988
Date:                Sun, 17 May 2026   AIC                            349.975
Time:                        05:00:40   BIC                            355.534
Sample:                             0   HQIC                           352.233
                                - 120                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ma.L1          0.3518      0.075      4.722      0.000       0.206       0.498
sigma2         1.0708      0.132      8.093      0.000       0.811       1.330
Ljung-Box (L1) (Q):                   0.01   Jarque-

## Candidate Comparison

ACF/PACF gives tentative candidates. AIC/BIC and residual checks help compare a small candidate set. Do not let a blind search replace model understanding: the chosen model still needs to make sense for the data and pass diagnostics.

In [6]:
search = arima_grid_search(
    y,
    p_values=(0, 1, 2),
    d_values=(0, 1),
    q_values=(0, 1, 2),
    trend_options=('n', 'c', 't'),
    max_results=12,
)
search

,order,trend,aic,bic,converged
0,"(0, 1, 1)",n,349.9755,355.5337,True
1,"(2, 1, 0)",n,350.7546,359.0920,True
2,"(1, 1, 2)",n,351.3063,362.4228,True
3,"(0, 1, 2)",n,351.9453,360.2826,True
4,"(1, 1, 1)",n,351.9592,360.2966,True
5,"(0, 1, 1)",t,351.9736,360.3110,True
6,"(1, 1, 0)",n,352.5736,358.1318,True
7,"(2, 1, 0)",t,352.7516,363.8681,True
8,"(2, 1, 1)",n,352.7534,363.8699,True
9,"(1, 1, 2)",t,353.3044,367.2000,True


Notice that `trend='c'` is not allowed when `d>0` in this `statsmodels` interface; a linear trend term represents drift after differencing. For classroom Box-Jenkins work, keep candidate searches small and interpretable.

In [7]:
forecast_table(ma1, steps=6).round(4)

,forecast,lower,upper
step,,,
1,15.8873,13.8592,17.9154
2,15.8873,12.4770,19.2976
3,15.8873,11.5116,20.2630
4,15.8873,10.7236,21.0510
5,15.8873,10.0408,21.7337
6,15.8873,9.4299,22.3447


In [8]:
plot_forecast(y, ma1, steps=6, title='ARIMA(0,1,1) forecast for towel sales', ylabel='Towel sales')

(<Figure size 900x450 with 1 Axes>,
 <Axes: title={'center': 'ARIMA(0,1,1) forecast for towel sales'}, xlabel='time', ylabel='Towel sales'>)

## One-Step Forecast Formula for ARIMA(0,1,1)

For the lecture convention

$$z_t = a_t - \theta_1 a_{t-1}, \quad z_t=y_t-y_{t-1},$$

all future random shocks have point prediction zero. Using the `statsmodels` convention $z_t=a_t+\psi_1a_{t-1}$, the next one-step point forecast is approximately

$$\hat y_{n+1|n}=y_n+\psi_1 e_n,$$

where $e_n$ is the last fitted residual and $\psi_1$ is the package MA coefficient. This is the package version of the hand calculation shown in the lecture/software notes.

In [9]:
psi1 = ma1.params.get('ma.L1')
last_resid = float(pd.Series(ma1.resid).iloc[-1])
manual_next = float(y.iloc[-1] + psi1 * last_resid)
package_next = float(forecast_table(ma1, steps=1)['forecast'].iloc[0])

pd.Series({
    'last_observed_y': float(y.iloc[-1]),
    'package_ma_L1_psi': float(psi1),
    'last_residual': last_resid,
    'manual_next_forecast': manual_next,
    'package_next_forecast': package_next,
    'absolute_difference': abs(manual_next - package_next),
}).round(6)

last_observed_y          15.6453
package_ma_L1_psi         0.3518
last_residual             0.6878
manual_next_forecast     15.8873
package_next_forecast    15.8873
absolute_difference       0.0000
dtype: float64

## A Stationary AR Example

The historical software notes also use viscosity data. Its plot is much more level-stationary than the towel sales series. The lecture example identifies an AR(2) candidate.

In [10]:
visc = pd.read_csv(DATA_DIR / 'viscosity_xb775.csv')
v = visc['Viscosity']
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
axes[0].plot(visc['Run'], v, marker='o')
axes[0].set_title('Viscosity series')
plot_acf(v, lags=18, ax=axes[1], title='ACF')
plot_pacf(v, lags=18, ax=axes[2], title='PACF', method='ywmle')
plt.tight_layout()

In [11]:
ar2 = fit_arima(v, order=(2, 0, 0), trend='c')
parameter_table(ar2).round(4)

,estimate,std_error,z_or_t,p_value
const,34.9464,0.3015,115.9042,0.0
ar.L1,0.6821,0.0902,7.5659,0.0
ar.L2,-0.4333,0.1044,-4.1523,0.0
sigma2,4.5436,0.5573,8.1534,0.0


## Practice Prompt

Suppose the raw series appears stationary around a nonzero level and PACF cuts off after lag 2 while ACF dies down. What would you fit first?

Reference idea: start with AR(2) with a constant, then check parameter estimates and residual diagnostics.